### <span style="color:#C80992"> NOTE : </span>    
* Does not run all cells blindly      
* In this we created lag feature column , new day/time column in main dataset      
* So avoid running that cells again because it can cause duplicate columns in main datset

## <span style="color:red"> Import Libraries </span>

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## <span style="color:red"> Loading Datasets </span>

In [3]:
DATA_PATH = Path("../data/processed/sales_audited.parquet")
sales = pd.read_parquet(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", sales.shape)

Dataset loaded successfully.
Shape: (7431525, 17)


In [4]:
sales.head()

,date,item_id,quantity,price_base,sum_total,store_id,dept_name,class_name,subclass_name,item_type,weight_volume,weight_netto,fatness,division,format,city,area
0,2023-08-04,293375605257,1.00,47.86,47.86,1,ПРИКАССА,ЖЕВАТЕЛЬНАЯ РЕЗИНКА,ПОДУШЕЧКИ,Подушечки,0.02,0.01,NaN,Div1,Format-1,City1,1500
1,2023-08-04,a66fdf2c0ae7,3.00,49.60,148.80,1,ЛИМОНАДЫ,НЕКОЛОСОДЕРЖАЩИЕ,"ДО 0,6Л.",Лимонады,0.50,0.50,NaN,Div1,Format-1,City1,1500
2,2023-08-04,daa46ef49b7a,0.82,379.00,311.54,1,СЛОЙКА,СЛОЙКА,СОБСТВЕННОЕ ПРОИЗВОДСТВО,Слойки,NaN,NaN,NaN,Div1,Format-1,City1,1500
3,2023-08-04,a3b49c1bf758,1.00,129.00,129.00,1,ПЕРСОНАЛЬНЫЙ УХОД,ДЛЯ ВАННЫ И ДУША,ЖИДКОЕ МЫЛО,Жидкое Мыло,0.20,0.20,NaN,Div1,Format-1,City1,1500
4,2023-08-04,ab611c5cef62,7.00,79.90,559.30,1,СОВРЕМЕННАЯ МОЛОЧНАЯ КАТЕГОРИЯ,ЙОГУРТЫ,ЙОГУРТЫ ЛОЖКОВЫЕ,Йогурты Ложковые,0.35,0.35,1.50,Div1,Format-1,City1,1500


In [5]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7431525 entries, 0 to 7431524
Data columns (total 17 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date           datetime64[ns]
 1   item_id        object        
 2   quantity       float64       
 3   price_base     float64       
 4   sum_total      float64       
 5   store_id       int64         
 6   dept_name      object        
 7   class_name     object        
 8   subclass_name  object        
 9   item_type      object        
 10  weight_volume  float64       
 11  weight_netto   float64       
 12  fatness        float64       
 13  division       object        
 14  format         object        
 15  city           object        
 16  area           int64         
dtypes: datetime64[ns](1), float64(6), int64(2), object(8)
memory usage: 963.9+ MB


<span style="color:brown"> **Sort the dataset correctly** </span>   
Before creating any time-series features, we'll establish the correct ordering

In [6]:
sales = sales.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

print("Data sorted by store → item → date.")

Data sorted by store → item → date.


## <span style="color:red"> Missing Values check </span>

In [7]:
sales.isna().sum()

date                   0
item_id                0
quantity               0
price_base             0
sum_total              0
store_id               0
dept_name          36580
class_name         36580
subclass_name      36580
item_type         691136
weight_volume    1435624
weight_netto     1114207
fatness          6778777
division               0
format                 0
city                   0
area                   0
dtype: int64

In [8]:
# Handle categorical missing values
categorical_missing_cols = [
    "dept_name",
    "class_name",
    "subclass_name",
    "item_type"
]

for col in categorical_missing_cols:
    sales[col] = sales[col].fillna("Unknown")

In [9]:
sales[categorical_missing_cols].isna().sum()

dept_name        0
class_name       0
subclass_name    0
item_type        0
dtype: int64

In [10]:
# Don't handle the numerical missing values yet.
# We'll first determine whether weight_volume, weight_netto, and especially fatness are actually useful for our forecasting model.

## <span style="color:red"> Date/Time Features </span>

In [11]:
sales["day_of_week"] = sales["date"].dt.dayofweek
sales["day_of_month"] = sales["date"].dt.day
sales["week_of_year"] = sales["date"].dt.isocalendar().week.astype("int8")
sales["month"] = sales["date"].dt.month
sales["quarter"] = sales["date"].dt.quarter
sales["year"] = sales["date"].dt.year
sales["is_weekend"] = (sales["day_of_week"] >= 5).astype("int8")

In [14]:
day_info = sales[["day_of_week","day_of_month","week_of_year","month","quarter","year","is_weekend"]]
day_info.head()

,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend
0,4,11,45,11,4,2022,0
1,6,13,45,11,4,2022,1
2,1,15,46,11,4,2022,0
3,1,22,47,11,4,2022,0
4,4,2,48,12,4,2022,0


In [15]:
day_info.describe()

,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend
count,7431525.00,7431525.00,7431525.00,7431525.00,7431525.00,7431525.00,7431525.00
mean,3.03,15.87,25.93,6.37,2.44,2023.36,0.28
std,1.97,8.79,14.53,3.34,1.07,0.69,0.45
min,0.00,1.00,1.00,1.00,1.00,2022.00,0.00
25%,1.00,8.00,14.00,4.00,2.00,2023.00,0.00
50%,3.00,16.00,26.00,6.00,2.00,2023.00,0.00
75%,5.00,23.00,37.00,9.00,3.00,2024.00,1.00
max,6.00,31.00,52.00,12.00,4.00,2024.00,1.00


In [16]:
date_range = sales["date"].agg(["min", "max"])

print("Overall date range:")
print("Start:", date_range["min"].date())
print("End  :", date_range["max"].date())

print("\nTotal calendar days:",
      (date_range["max"] - date_range["min"]).days + 1)

Overall date range:
Start: 2022-08-28
End  : 2024-09-26

Total calendar days: 761


## <span style="color:red"> Checking items </span>

In [17]:
# Check whether each Item–Store has continuous dates
# For a particular item_id + store_id, does it have a record for every day, or are some calendar days missing?

item_store_dates = (
    sales.groupby(["item_id", "store_id"])["date"]
    .agg(["min", "max", "count"])
    .reset_index()
)

item_store_dates["expected_days"] = (
    item_store_dates["max"] - item_store_dates["min"]
).dt.days + 1

item_store_dates["missing_days"] = (
    item_store_dates["expected_days"] - item_store_dates["count"]
)

item_store_dates.head()

,item_id,store_id,min,max,count,expected_days,missing_days
0,00179dda14f8,1,2022-11-11,2023-08-05,22,268,246
1,001829cb707d,1,2022-08-28,2024-09-26,626,761,135
2,001829cb707d,2,2022-08-28,2024-09-25,473,760,287
3,001829cb707d,3,2022-08-30,2022-11-08,35,71,36
4,001829cb707d,4,2023-12-17,2024-09-23,219,282,63


In [18]:
print("Total item-store combinations:",
      len(item_store_dates))

print(
    "Item-store combinations with missing calendar days:",
    (item_store_dates["missing_days"] > 0).sum()
)

print(
    "Item-store combinations with complete daily records:",
    (item_store_dates["missing_days"] == 0).sum()
)

Total item-store combinations: 58025
Item-store combinations with missing calendar days: 55125
Item-store combinations with complete daily records: 2900


In [20]:
# Measure activity of item-store combinations
item_store_activity = (
    sales.groupby(["item_id", "store_id"])
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        active_days=("date", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

item_store_activity["span_days"] = (
    item_store_activity["last_date"]
    - item_store_activity["first_date"]
).dt.days + 1

item_store_activity["coverage_pct"] = (
    item_store_activity["active_days"]
    / item_store_activity["span_days"]
    * 100
)

item_store_activity.head()

,item_id,store_id,first_date,last_date,active_days,total_quantity,span_days,coverage_pct
0,00179dda14f8,1,2022-11-11,2023-08-05,22,22.00,268,8.21
1,001829cb707d,1,2022-08-28,2024-09-26,626,2048.00,761,82.26
2,001829cb707d,2,2022-08-28,2024-09-25,473,868.00,760,62.24
3,001829cb707d,3,2022-08-30,2022-11-08,35,85.00,71,49.30
4,001829cb707d,4,2023-12-17,2024-09-23,219,584.00,282,77.66


In [21]:
item_store_activity["coverage_pct"].describe()

count   58025.00
mean       39.02
std        31.02
min         0.29
25%        11.63
50%        30.64
75%        62.79
max       100.00
Name: coverage_pct, dtype: float64

In [22]:
print("Item-store combinations:", len(item_store_activity))

print(
    "Combinations with >= 25% daily coverage:",
    (item_store_activity["coverage_pct"] >= 25).sum()
)

print(
    "Combinations with >= 50% daily coverage:",
    (item_store_activity["coverage_pct"] >= 50).sum()
)

print(
    "Combinations with >= 75% daily coverage:",
    (item_store_activity["coverage_pct"] >= 75).sum()
)

Item-store combinations: 58025
Combinations with >= 25% daily coverage: 32615
Combinations with >= 50% daily coverage: 19698
Combinations with >= 75% daily coverage: 10339


## <span style="color:red"> Lag Features </span>

In [23]:
# Before creating the actual lag features, let's create a lookup table containing the existing daily demand.
quantity_lookup = sales[
    ["date", "item_id", "store_id", "quantity"]
].copy()

quantity_lookup = quantity_lookup.rename(
    columns={"quantity": "past_quantity"}
)

print("Quantity lookup created.")
print("Shape:", quantity_lookup.shape)

Quantity lookup created.
Shape: (7431525, 4)


In [24]:
# Let's start with lag_1 only and verify that the logic works before creating all lags.
lag_1_lookup = quantity_lookup.copy()

lag_1_lookup["date"] = (
    lag_1_lookup["date"] + pd.Timedelta(days=1)
)

lag_1_lookup = lag_1_lookup.rename(
    columns={"past_quantity": "lag_1"}
)

sales = sales.merge(
    lag_1_lookup,
    on=["date", "item_id", "store_id"],
    how="left",
    validate="one_to_one"
)

sales["lag_1"] = sales["lag_1"].fillna(0)

print("lag_1 created successfully.")

lag_1 created successfully.


In [29]:
sales[["date", "item_id", "store_id", "quantity", "lag_1"]].head(20)

,date,item_id,store_id,quantity,lag_1
0,2022-11-11,00179dda14f8,1,1.00,0.00
1,2022-11-13,00179dda14f8,1,1.00,0.00
2,2022-11-15,00179dda14f8,1,1.00,0.00
3,2022-11-22,00179dda14f8,1,1.00,0.00
4,2022-12-02,00179dda14f8,1,1.00,0.00
5,2022-12-06,00179dda14f8,1,1.00,0.00
6,2022-12-12,00179dda14f8,1,1.00,0.00
7,2022-12-29,00179dda14f8,1,1.00,0.00
8,2022-12-31,00179dda14f8,1,1.00,0.00
9,2023-01-08,00179dda14f8,1,1.00,0.00


#### <span style="color:brown"> Create remaining lag features </span>
* lag_1 → previous day
* lag_7 → previous 7 days
* lag_14 → previous 14 days
* lag_28 → previous 28 days

In [30]:
for lag in [7, 14, 28]:
    
    lag_lookup = sales[
        ["date", "item_id", "store_id", "quantity"]
    ].copy()

    lag_lookup["date"] = (
        lag_lookup["date"] + pd.Timedelta(days=lag)
    )

    lag_lookup = lag_lookup.rename(
        columns={"quantity": f"lag_{lag}"}
    )

    sales = sales.merge(
        lag_lookup,
        on=["date", "item_id", "store_id"],
        how="left",
        validate="one_to_one"
    )

    sales[f"lag_{lag}"] = sales[f"lag_{lag}"].fillna(0)

    del lag_lookup

    print(f"lag_{lag} created.")

lag_7 created.
lag_14 created.
lag_28 created.


In [31]:
# Checking all lag features
sales[["lag_1", "lag_7", "lag_14", "lag_28"]].isna().sum()

lag_1     0
lag_7     0
lag_14    0
lag_28    0
dtype: int64

## <span style="color:red"> Rolling Demand Features </span>

In [32]:
# Free unnecessary memory
del quantity_lookup
print("Temporary quantity lookup removed from memory.")

Temporary quantity lookup removed from memory.


In [40]:
# Create a function for calendar-aware rolling mean
# This function calculates the average demand over the previous N calendar days, including zero-demand days.

def calendar_rolling_mean(df, window):
    result = pd.Series(0.0, index=df.index)

    for (item, store), group in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ):
        group = group.sort_values("date")

        dates = group["date"]
        quantities = group["quantity"]

        cumulative = quantities.cumsum()
        cumulative_before = cumulative - quantities

        # Start of the rolling window
        boundary_dates = dates - pd.Timedelta(days=window)

        boundary_df = pd.DataFrame({
            "date": dates.values,
            "cumulative": cumulative.values
        }).sort_values("date")

        lookup_df = pd.DataFrame({
            "date": boundary_dates.values
        }).sort_values("date")

        boundary_values = pd.merge_asof(
            lookup_df,
            boundary_df,
            on="date",
            direction="backward"
        )["cumulative"].fillna(0).to_numpy()

        rolling_sum = (
            cumulative_before.to_numpy()
            - boundary_values
        )

        result.loc[group.index] = rolling_sum / window

    return result

In [41]:
sales["rolling_mean_7"] = calendar_rolling_mean(sales,7)
print("Correct rolling_mean_7 created.")

Correct rolling_mean_7 created.


In [42]:
sales[["date","item_id","store_id","quantity","lag_1","rolling_mean_7"]].head(20)

,date,item_id,store_id,quantity,lag_1,rolling_mean_7
0,2022-11-11,00179dda14f8,1,1.00,0.00,0.00
1,2022-11-13,00179dda14f8,1,1.00,0.00,0.14
2,2022-11-15,00179dda14f8,1,1.00,0.00,0.29
3,2022-11-22,00179dda14f8,1,1.00,0.00,0.00
4,2022-12-02,00179dda14f8,1,1.00,0.00,0.00
5,2022-12-06,00179dda14f8,1,1.00,0.00,0.14
6,2022-12-12,00179dda14f8,1,1.00,0.00,0.14
7,2022-12-29,00179dda14f8,1,1.00,0.00,0.00
8,2022-12-31,00179dda14f8,1,1.00,0.00,0.14
9,2023-01-08,00179dda14f8,1,1.00,0.00,0.00


In [44]:
# 14-day and 28-day rolling
sales["rolling_mean_14"] = calendar_rolling_mean(sales,14)
print("rolling_mean_14 created.")

sales["rolling_mean_28"] = calendar_rolling_mean(sales,28)
print("rolling_mean_28 created.")

rolling_mean_14 created.
rolling_mean_28 created.


In [45]:
# Verify all rolling features
sales[[
        "date",
        "item_id",
        "store_id",
        "quantity",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ]].head(20)

,date,item_id,store_id,quantity,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,2022-11-11,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,2022-11-13,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.14,0.07,0.04
2,2022-11-15,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.29,0.14,0.07
3,2022-11-22,00179dda14f8,1,1.00,0.00,1.00,0.00,0.00,0.00,0.21,0.11
4,2022-12-02,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.00,0.07,0.14
5,2022-12-06,00179dda14f8,1,1.00,0.00,0.00,1.00,0.00,0.14,0.07,0.18
6,2022-12-12,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.14,0.14,0.14
7,2022-12-29,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.11
8,2022-12-31,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.14,0.07,0.11
9,2023-01-08,00179dda14f8,1,1.00,0.00,0.00,0.00,0.00,0.00,0.14,0.11


In [46]:
# Checking Missing Values 
sales[[
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ]].isna().sum()

rolling_mean_7     0
rolling_mean_14    0
rolling_mean_28    0
dtype: int64

## <span style="color:red"> Seasonal Features </span>

In [47]:
# Create cyclical seasonal features
sales["month_sin"] = np.sin(
    2 * np.pi * sales["month"] / 12
)

sales["month_cos"] = np.cos(
    2 * np.pi * sales["month"] / 12
)

sales["dow_sin"] = np.sin(
    2 * np.pi * sales["day_of_week"] / 7
)

sales["dow_cos"] = np.cos(
    2 * np.pi * sales["day_of_week"] / 7
)

print("Cyclical seasonal features created.")

Cyclical seasonal features created.


In [48]:
sales[
    [
        "date",
        "month",
        "month_sin",
        "month_cos",
        "day_of_week",
        "dow_sin",
        "dow_cos"
    ]
].head(10)

,date,month,month_sin,month_cos,day_of_week,dow_sin,dow_cos
0,2022-11-11,11,-0.50,0.87,4,-0.43,-0.90
1,2022-11-13,11,-0.50,0.87,6,-0.78,0.62
2,2022-11-15,11,-0.50,0.87,1,0.78,0.62
3,2022-11-22,11,-0.50,0.87,1,0.78,0.62
4,2022-12-02,12,-0.00,1.00,4,-0.43,-0.90
5,2022-12-06,12,-0.00,1.00,1,0.78,0.62
6,2022-12-12,12,-0.00,1.00,0,0.00,1.00
7,2022-12-29,12,-0.00,1.00,3,0.43,-0.90
8,2022-12-31,12,-0.00,1.00,5,-0.97,-0.22
9,2023-01-08,1,0.50,0.87,6,-0.78,0.62


## <span style="color:red"> Leakage & Feature Validation </span>

In [49]:
print("Quantity missing:", sales["quantity"].isna().sum())
print("Negative quantity:", (sales["quantity"] < 0).sum())
print("Quantity dtype:", sales["quantity"].dtype)

Quantity missing: 0
Negative quantity: 0
Quantity dtype: float64


In [50]:
engineered_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos"
]

sales[engineered_features].isna().sum()

lag_1              0
lag_7              0
lag_14             0
lag_28             0
rolling_mean_7     0
rolling_mean_14    0
rolling_mean_28    0
month_sin          0
month_cos          0
dow_sin            0
dow_cos            0
dtype: int64

In [51]:
# Check infinite values
np.isinf(
    sales[engineered_features]
).sum()

lag_1              0
lag_7              0
lag_14             0
lag_28             0
rolling_mean_7     0
rolling_mean_14    0
rolling_mean_28    0
month_sin          0
month_cos          0
dow_sin            0
dow_cos            0
dtype: int64

## <span style="color:red"> Final preparation check </span>

In [52]:
print("Final shape:", sales.shape)

print("\nFinal columns:")
for i, col in enumerate(sales.columns, 1):
    print(f"{i}. {col}")

Final shape: (7431525, 35)

Final columns:
1. date
2. item_id
3. quantity
4. price_base
5. sum_total
6. store_id
7. dept_name
8. class_name
9. subclass_name
10. item_type
11. weight_volume
12. weight_netto
13. fatness
14. division
15. format
16. city
17. area
18. day_of_week
19. day_of_month
20. week_of_year
21. month
22. quarter
23. year
24. is_weekend
25. lag_1
26. lag_7
27. lag_14
28. lag_28
29. rolling_mean_7
30. rolling_mean_14
31. rolling_mean_28
32. month_sin
33. month_cos
34. dow_sin
35. dow_cos


In [53]:
# Check key uniqueness again
key_columns = ["date", "item_id", "store_id"]

duplicate_count = sales.duplicated(
    subset=key_columns
).sum()

print("Duplicate Date + Item + Store combinations:", duplicate_count)

Duplicate Date + Item + Store combinations: 0


In [54]:
# Final missing-value summary
remaining_missing = sales.isna().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)

remaining_missing

fatness          6778777
weight_volume    1435624
weight_netto     1114207
dtype: int64

## <span style="color:red"> Final feature summary </span>

In [55]:
feature_groups = {
    "Target": [
        "quantity"
    ],
    
    "Identifiers": [
        "date",
        "item_id",
        "store_id"
    ],
    
    "Business / Product Features": [
        "price_base",
        "sum_total",
        "dept_name",
        "class_name",
        "subclass_name",
        "item_type",
        "weight_volume",
        "weight_netto",
        "fatness",
        "division",
        "format",
        "city",
        "area"
    ],
    
    "Date Features": [
        "day_of_week",
        "day_of_month",
        "week_of_year",
        "month",
        "quarter",
        "year",
        "is_weekend"
    ],
    
    "Lag Features": [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28"
    ],
    
    "Rolling Features": [
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ],
    
    "Seasonal Features": [
        "month_sin",
        "month_cos",
        "dow_sin",
        "dow_cos"
    ]
}

for group, columns in feature_groups.items():
    print(f"\n{group} ({len(columns)} features)")
    print(columns)


Target (1 features)
['quantity']

Identifiers (3 features)
['date', 'item_id', 'store_id']

Business / Product Features (13 features)
['price_base', 'sum_total', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'weight_volume', 'weight_netto', 'fatness', 'division', 'format', 'city', 'area']

Date Features (7 features)
['day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend']

Lag Features (4 features)
['lag_1', 'lag_7', 'lag_14', 'lag_28']

Rolling Features (3 features)
['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']

Seasonal Features (4 features)
['month_sin', 'month_cos', 'dow_sin', 'dow_cos']


In [56]:
print("Rows:", len(sales))
print("Columns:", len(sales.columns))

print("\nQuantity missing:", sales["quantity"].isna().sum())
print("Negative quantity:", (sales["quantity"] < 0).sum())

print(
    "Duplicate Date + Item + Store:",
    sales.duplicated(
        subset=["date", "item_id", "store_id"]
    ).sum()
)

print(
    "Infinite numerical values:",
    np.isinf(
        sales.select_dtypes(include=np.number)
    ).sum().sum()
)

Rows: 7431525
Columns: 35

Quantity missing: 0
Negative quantity: 0
Duplicate Date + Item + Store: 0
Infinite numerical values: 0


## <span style="color:red"> Save as Parquet </span>

In [57]:
OUTPUT_PATH = Path("../data/processed/sales_prepared.parquet")

sales.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Prepared dataset saved successfully.")
print("Path:", OUTPUT_PATH)

Prepared dataset saved successfully.
Path: ..\data\processed\sales_prepared.parquet


In [58]:
print("File exists:", OUTPUT_PATH.exists())

if OUTPUT_PATH.exists():
    print(
        "File size:",
        round(OUTPUT_PATH.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

File exists: True
File size: 147.51 MB
